# Section 1: Environment Setup, Paths & Data Loading

In [5]:
import os
import joblib
import pandas as pd
import numpy as np
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
# 2. Define path to your project folder in Google Drive
# Update 'Telecom_Cell_Outage_Project' to your exact Drive folder name
# PROJECT_DIR should be a directory, not a file.
PROJECT_DIR = '/content/drive/MyDrive'

# DATA_PATH should now directly point to your dataset file.
DATA_PATH = os.path.join(PROJECT_DIR, 'mobi_data2_final_engineered.csv')

# MODELS_DIR will now correctly be '/content/drive/MyDrive/models'
MODELS_DIR = os.path.join(PROJECT_DIR, 'models')

os.makedirs(MODELS_DIR, exist_ok=True)

# 3. Load Featured Dataset
print(f"Loading dataset from: {DATA_PATH}")
if DATA_PATH.endswith('.parquet'):
    df = pd.read_parquet(DATA_PATH)
else:
    df = pd.read_csv(DATA_PATH)

print(f"Dataset successfully loaded. Shape: {df.shape}")

Loading dataset from: /content/drive/MyDrive/mobi_data2_final_engineered.csv
Dataset successfully loaded. Shape: (999999, 23)


# Section 2: Stratified Split & Multi-Model Training Pipeline


In [14]:
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
import pandas as pd

# 1. Separate Features (X) and Target (y)
metadata_to_drop = [
    'location_information', 'additional_information',
    'enodeb_id', 'gnodeb_id', 'mo_name', 'alarm_id'
]

X = df.drop(columns=[c for c in metadata_to_drop if c in df.columns], errors='ignore')
if 'target' in X.columns:
    y = X['target']
    X = X.drop(columns=['target'])
else:
    raise KeyError("Target column 'target' not found in dataset!")

# Preserve categorical types
for col in X.select_dtypes(include=['object']).columns:
    X[col] = X[col].astype('category')

# 2. Stratified 80/20 Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Compute class balance scale factor (~1:9 imbalance ratio)
scale_pos = (len(y_train) - sum(y_train)) / sum(y_train)
print(f"Training Features: {X_train.shape} | Testing Features: {X_test.shape}")
print(f"Calculated Imbalance Ratio (scale_pos_weight): {scale_pos:.4f}\n")

# 3. Define Model Suite
models = {
    'XGBoost': XGBClassifier(
        n_estimators=300, learning_rate=0.03, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=scale_pos, enable_categorical=True,
        tree_method='hist', random_state=42, n_jobs=-1
    ),
    'LightGBM': LGBMClassifier(
        n_estimators=300, learning_rate=0.03, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=scale_pos, random_state=42, n_jobs=-1, verbose=-1
    ),
    'RandomForest': RandomForestClassifier(
        n_estimators=200, max_depth=12, class_weight='balanced',
        random_state=42, n_jobs=-1
    )
}

# 4. Train Models & Measure Validation Benchmarks
trained_models = {}
validation_summary = []

print("--- Starting Multi-Model Comparison Training ---")
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)

    y_probs = model.predict_proba(X_test)[:, 1]

    roc = roc_auc_score(y_test, y_probs)
    pr_auc = average_precision_score(y_test, y_probs)

    trained_models[name] = model
    validation_summary.append({
        'Model': name,
        'ROC-AUC': round(roc, 4),
        'PR-AUC': round(pr_auc, 4)
    })
    print(f"  └─ {name} | ROC-AUC: {roc:.4f} | PR-AUC: {pr_auc:.4f}")

# 5. Display Leaderboard Summary
summary_df = pd.DataFrame(validation_summary).sort_values(by='PR-AUC', ascending=False)
print("\n" + "="*45)
print("     MULTI-MODEL TRAINING LEADERBOARD      ")
print("="*45)
print(summary_df.to_string(index=False))

# 6. Save Model Artifacts locally in your repo's /models directory
joblib.dump(trained_models, os.path.join(MODELS_DIR, 'all_trained_models.joblib'))
joblib.dump({'X_test': X_test, 'y_test': y_test}, os.path.join(MODELS_DIR, 'test_data_split.joblib'))

print(f"\nAll artifacts successfully saved to local directory: {MODELS_DIR}/")

Training Features: (799999, 22) | Testing Features: (200000, 22)
Calculated Imbalance Ratio (scale_pos_weight): 8.3985

--- Starting Multi-Model Comparison Training ---
Training XGBoost...
  └─ XGBoost | ROC-AUC: 0.9791 | PR-AUC: 0.8930
Training LightGBM...
  └─ LightGBM | ROC-AUC: 0.9794 | PR-AUC: 0.8943
Training RandomForest...
  └─ RandomForest | ROC-AUC: 0.9756 | PR-AUC: 0.8795

     MULTI-MODEL TRAINING LEADERBOARD      
       Model  ROC-AUC  PR-AUC
    LightGBM   0.9794  0.8943
     XGBoost   0.9791  0.8930
RandomForest   0.9756  0.8795

All artifacts successfully saved to local directory: /content/drive/MyDrive/models/
